<a href="https://colab.research.google.com/github/dineshaimldev/Computer-Vision/blob/main/simple_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3

CLASS_NAMES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

In [ ]:
import pathlib
import os

dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir_name = tf.keras.utils.get_file(
    'flower_photos',
    origin=dataset_url,
    untar=True,
    cache_dir='.'
)

data_dir = pathlib.Path(data_dir_name) / 'flower_photos'

print(f"Dataset extracted to: {data_dir}")

In [ ]:
global CLASS_NAMES
CLASS_NAMES = sorted([item.name for item in data_dir.glob('*') if item.name != "LICENSE.txt"])

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=16,
    shuffle=True,
    seed=123,
    validation_split=0.2,
    subset='training',
    class_names=CLASS_NAMES
)

eval_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=16,
    shuffle=True,
    seed=123,
    validation_split=0.2,
    subset='validation',
    class_names=CLASS_NAMES
)

def normalize_img(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

train_dataset = train_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
eval_dataset = eval_ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print(f"Inferred CLASS_NAMES: {CLASS_NAMES}")

In [ ]:
for image_batch, label_batch in train_dataset.take(3):
    print("Image batch shape:", image_batch.shape)
    print("Label batch shape:", label_batch.shape)
    print("Labels:", label_batch.numpy())

In [ ]:
import matplotlib.pyplot as plt

for image_batch, label_batch in train_dataset.take(2):
    first_image = image_batch[0]
    first_label = label_batch[0]

    plt.imshow(first_image.numpy())
    plt.title(f"Label: {CLASS_NAMES[first_label]}")
    plt.axis('off')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

for image_batch, label_batch in train_dataset.take(2):
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))

    for i in range(16):
        ax = axes[i // 4, i % 4]
        ax.imshow(image_batch[i].numpy())
        ax.set_title(f"Label: {CLASS_NAMES[label_batch[i]]}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),
    keras.layers.Dense(len(CLASS_NAMES), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

In [ ]:
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=eval_dataset,
    epochs=EPOCHS
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

for images, labels in eval_dataset.take(1):
    batch_predictions = model.predict(images)
    predicted_indices = np.argmax(batch_predictions, axis=1)

    num_images = images.shape[0]

    num_cols = 4
    num_rows = math.ceil(num_images / num_cols)

    plt.figure(figsize=(12, 3 * num_rows))

    for i in range(num_images):
        plt.subplot(num_rows, num_cols, i + 1)

        plt.imshow(images[i].numpy())
        plt.axis('off')

        pred_class = CLASS_NAMES[predicted_indices[i]]
        actual_class = CLASS_NAMES[labels[i].numpy()]

        plt.title(f"Pred: {pred_class}\nActual: {actual_class}", fontsize=10)

    plt.tight_layout()
    plt.show()